<a href="https://colab.research.google.com/github/xingji1337/week5RAG/blob/TrackA_Rerank_ContextOpt/Week5_1_RAG_Rerank_ContextOpt_HOME_REPAIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 — Track A: Rerank & Context Optimization (Home Repair AI)

This notebook builds a **retrieval-augmented generation (RAG)** pipeline for a *Home Repair Assistant* using your uploaded PDFs as the knowledge corpus. It includes:
- **BM25 + Dense** retrieval with **RRF fusion**  
- Optional **cross-encoder reranker**  
- **MMR** (diversity) and **context compression** (token budget aware)  
- Light **logging**: recall-like signal, latency, context length, (optional) token cost  
- 2–3 qualitative examples (wins/failures)

> Data domain: Home Repair (drywall, water leaks, complete home repair guide).  
> Sources: `Complete home repair  with 350 projects and 2300 photos.pdf`, `7 Different Ways to Repair Drywall.pdf`, `How to stop Water damage when A Leak.pdf`


## 0) Install dependencies (Colab/Local)

In [1]:
# If running in Colab, uncomment:
!pip -q install sentence-transformers rank_bm25 faiss-cpu pypdf rapidfuzz transformers accelerate bitsandbytes

# If FAISS install fails on your platform, you can use a simple cosine search fallback.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 19.4 MB/s eta 0:00:00


## 1) Mount Google Drive (optional) & set paths

In [2]:
import os, sys, time, json, math, random, pathlib, re
from typing import List, Dict, Any, Tuple

# If using Colab and your PDFs live in Drive, mount then point DATA_DIR to your folder.
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Drive mounted.")
    except Exception as e:
        print("Colab not detected or mount failed:", e)

# Default to the uploaded files path in this environment:
# DATA_DIR = '/mnt/data' # Original path

# Update DATA_DIR to point to the correct directory containing the uploaded PDF files
DATA_DIR = '/content/drive/MyDrive/' # Assuming the PDFs are in the default Colab content directory

PDF_FILES = [
    os.path.join(DATA_DIR, 'Complete home repair  with 350 projects and 2300 photos.pdf'),
    os.path.join(DATA_DIR, '7 Different Ways to Repair Drywall.pdf'),
    os.path.join(DATA_DIR, 'How to stop Water damage when A Leak.pdf')
]

for p in PDF_FILES:
    print("Exists?", os.path.exists(p), p)

Mounted at /content/drive
Drive mounted.
Exists? True /content/drive/MyDrive/Complete home repair  with 350 projects and 2300 photos.pdf
Exists? True /content/drive/MyDrive/7 Different Ways to Repair Drywall.pdf
Exists? True /content/drive/MyDrive/How to stop Water damage when A Leak.pdf


## 2) PDF loading, chunking, and indexing utils

In [3]:
# Minimal PDF text loader using pypdf
from dataclasses import dataclass

try:
    from pypdf import PdfReader
except Exception as e:
    PdfReader = None
    print("pypdf not available; please `pip install pypdf` before running.")

def load_pdf_text(path: str) -> List[str]:
    pages = []
    if PdfReader is None:
        raise RuntimeError("pypdf not installed. Run the install cell first.")
    reader = PdfReader(path)
    for i, page in enumerate(reader.pages):
        try:
            txt = page.extract_text() or ""
        except Exception:
            txt = ""
        if txt.strip():
            pages.append(txt)
    return pages

def chunk_text(text: str, chunk_size: int = 800, chunk_overlap: int = 150) -> List[str]:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+chunk_size]
        chunks.append(" ".join(chunk))
        i += (chunk_size - chunk_overlap)
    return chunks

@dataclass
class DocChunk:
    doc_id: str
    chunk_id: int
    text: str
    source: str
    meta: Dict[str, Any]

def build_corpus(pdf_files: List[str], chunk_size=800, chunk_overlap=150) -> List[DocChunk]:
    corpus = []
    for src in pdf_files:
        doc_id = pathlib.Path(src).stem
        pages = load_pdf_text(src)
        full = "\n\n".join(pages)
        chunks = chunk_text(full, chunk_size, chunk_overlap)
        for idx, ch in enumerate(chunks):
            corpus.append(DocChunk(
                doc_id=doc_id,
                chunk_id=idx,
                text=ch,
                source=src,
                meta={"doc": doc_id, "chunk": idx}
            ))
    print(f"Built corpus with {len(corpus)} chunks from {len(pdf_files)} PDFs.")
    return corpus

corpus = build_corpus(PDF_FILES)


Built corpus with 348 chunks from 3 PDFs.


## 3) BM25 retriever

In [4]:
from rank_bm25 import BM25Okapi

tokenized = [c.text.split() for c in corpus]
bm25 = BM25Okapi(tokenized)

def bm25_search(query: str, k: int = 20) -> List[Tuple[float, DocChunk]]:
    scores = bm25.get_scores(query.split())
    ranked = sorted(list(enumerate(scores)), key=lambda x: x[1], reverse=True)[:k]
    return [(score, corpus[idx]) for idx, score in ranked]


## 4) Dense retriever (Sentence Transformers)

In [5]:
try:
    from sentence_transformers import SentenceTransformer, util
    _ST_AVAILABLE = True
except Exception as e:
    _ST_AVAILABLE = False
    print("sentence-transformers not available; install first to use dense retrieval.")

dense_model_name = "sentence-transformers/all-MiniLM-L6-v2"  # fast & light
dense_model = None
dense_embeddings = None

def ensure_dense():
    global dense_model, dense_embeddings
    if not _ST_AVAILABLE:
        raise RuntimeError("Install sentence-transformers to enable dense retrieval.")
    if dense_model is None:
        dense_model = SentenceTransformer(dense_model_name)
        texts = [c.text for c in corpus]
        dense_embeddings = dense_model.encode(texts, convert_to_tensor=True, show_progress_bar=True)
        print("Dense index built.")
    return dense_model, dense_embeddings

def dense_search(query: str, k: int = 20) -> List[Tuple[float, DocChunk]]:
    model, embs = ensure_dense()
    q = model.encode([query], convert_to_tensor=True)
    cos = util.cos_sim(q, embs)[0]
    topk = int(min(k, len(corpus)))
    vals, idxs = cos.topk(topk)
    out = []
    for score, idx in zip(vals.tolist(), idxs.tolist()):
        out.append((float(score), corpus[idx]))
    return out


## 5) Reciprocal Rank Fusion (RRF)

In [6]:
from collections import defaultdict

def rrf_fuse(results_lists: List[List[DocChunk]], K: float = 60.0, topk: int = 20) -> List[Tuple[float, DocChunk]]:
    # results_lists: list of ranked lists [(score, chunk), ...]
    rrfs = defaultdict(float)
    ranks = defaultdict(dict)

    for rl_idx, rl in enumerate(results_lists):
        for rank, (score, ch) in enumerate(rl, start=1):
            key = (ch.doc_id, ch.chunk_id)
            rrfs[key] += 1.0 / (K + rank)
            ranks[key] = ch

    fused = sorted([(score, ranks[key]) for key, score in rrfs.items()],
                   key=lambda x: x[0], reverse=True)[:topk]
    return fused

def retrieve_rrf(query: str, k_each: int = 20, topk: int = 20) -> List[Tuple[float, DocChunk]]:
    b = bm25_search(query, k=k_each)
    try:
        d = dense_search(query, k=k_each)
    except Exception:
        d = []
    return rrf_fuse([b, d], K=60.0, topk=topk)


## 6) Optional reranker (Cross-Encoder)

In [7]:
# Light cross-encoder (optional). If unavailable, skip.
RERANK = False
cross_encoder = None
try:
    if RERANK:
        from sentence_transformers import CrossEncoder
        cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
except Exception as e:
    print("Cross-encoder not available yet. Set RERANK=True after installing sentence-transformers.")

def rerank(query: str, candidates: List[Tuple[float, DocChunk]], topk: int = 20):
    if not RERANK or cross_encoder is None:
        return candidates[:topk]
    pairs = [(query, ch.text) for _, ch in candidates]
    scores = cross_encoder.predict(pairs)
    rescored = sorted([(float(s), ch) for s, (_, ch) in zip(scores, candidates)], key=lambda x: x[0], reverse=True)
    return rescored[:topk]


## 7) Maximal Marginal Relevance (MMR) for diversity

In [8]:
import numpy as np
def mmr(query: str, ranked: List[Tuple[float, DocChunk]], lambda_mult: float = 0.7, topk: int = 10):
    # requires dense model to compute repulsions; if absent, returns topk
    if not _ST_AVAILABLE:
        return ranked[:topk]
    model, _ = ensure_dense()
    q_vec = model.encode([query], convert_to_tensor=False, normalize_embeddings=True)[0]
    cand_texts = [ch.text for _, ch in ranked]
    cand_vecs = model.encode(cand_texts, convert_to_tensor=False, normalize_embeddings=True)

    selected, selected_idx = [], []
    remaining = list(range(len(cand_vecs)))

    while remaining and len(selected) < topk:
        best_idx, best_score = None, -1e9
        for i in remaining:
            rel = np.dot(q_vec, cand_vecs[i])
            div = 0.0 if not selected_idx else max(np.dot(cand_vecs[i], cand_vecs[j]) for j in selected_idx)
            score = lambda_mult * rel - (1 - lambda_mult) * div
            if score > best_score:
                best_score, best_idx = score, i
        selected_idx.append(best_idx)
        remaining.remove(best_idx)
        selected.append(ranked[best_idx])
    return selected


## 8) Context compression (token budget keeper)

In [9]:
# Simple heuristic compressor: keep top-N sentences v2 using RapidFuzz for sentence scoring
from rapidfuzz import fuzz

def compress_chunk(query: str, text: str, max_chars: int = 900) -> str:
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    scored = [(fuzz.partial_ratio(query, s), s) for s in sents]
    scored.sort(key=lambda x: x[0], reverse=True)
    out = ""
    for sc, s in scored:
        if len(out) + len(s) + 1 > max_chars:
            break
        out += (" " if out else "") + s
    return out if out else text[:max_chars]

def build_context(query: str, ranked: List[Tuple[float, DocChunk]], max_context_chars: int = 3500):
    ctx_parts = []
    total = 0
    for _, ch in ranked:
        comp = compress_chunk(query, ch.text, max_chars=900)
        tag = f"[{ch.doc_id}#${ch.chunk_id}]"
        seg = f"{tag} {comp}"
        if total + len(seg) + 2 > max_context_chars:
            break
        ctx_parts.append(seg)
        total += len(seg) + 2
    return "\n\n".join(ctx_parts)


## 9) Answer generator (LLM stub with citation enforcement)

In [10]:
# For portability, we won't call a remote LLM here. We simulate formatting.
def answer_with_context(query: str, context: str) -> str:
    # In production: call your LLM with a prompt that *requires* inline citations like [doc_id#$chunk].
    return f"""Q: {query}

Relevant context (truncated):
{context[:700]}

A (draft): Based on the retrieved context above, here is a grounded answer with inline citations.
"""

def run_pipeline(query: str, k_each=20, topk=15, lambda_mmr=0.7):
    t0 = time.time()
    fused = retrieve_rrf(query, k_each=k_each, topk=topk)
    fused = rerank(query, fused, topk=topk)
    fused = mmr(query, fused, lambda_mult=lambda_mmr, topk=min(10, topk))
    context = build_context(query, fused, max_context_chars=3500)
    t1 = time.time()
    ans = answer_with_context(query, context)
    logs = {
        "latency_s": round(t1 - t0, 3),
        "num_ctx_chunks": context.count('\n\n') + 1 if context else 0,
        "ctx_chars": len(context)
    }
    return ans, context, logs, fused

example_query = "How do I repair a small doorknob hole in drywall and what safety steps should I take?"
ans, ctx, logs, fused = run_pipeline(example_query)
print(ans)
print("\nLogs:", logs)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Dense index built.
Q: How do I repair a small doorknob hole in drywall and what safety steps should I take?

Relevant context (truncated):
[7 Different Ways to Repair Drywall#$1] When dry, sand lightly, then prime and paint. Step 2: Use a four- or six-inch-wide drywall knife to apply joint compound over the patch. Corner bead is nailed over the corner and then concealed by two or three layers of joint compound. Problem 1: Doorknob Damage Step 1: One of the most common drywall repairs occurs when a door is swung open a little too forcefully and the doorknob punches a hole through the drywall. ⚒️ Problem 2: Crumpled Corner Bead Step 1: When two sheets of drywall meet at an outside wall corner, they're protected by an L-shaped metal strip called a corner bead. For smaller repairs, something like this will suffice. The good news is

A (draft): Based on the retrieved context above, here is a grounded answer with inline citations.


Logs: {'latency_s': 12.002, 'num_ctx_chunks': 3, 'ctx_chars

## 10) Ablation: Baseline vs +Rerank vs +Compression vs +Both

In [11]:
import pandas as pd

def baseline_only(query):
    b = bm25_search(query, k=20)
    ctx = build_context(query, b, max_context_chars=3500)  # no compression knob change here
    return ctx, b

def with_rerank_only(query):
    f = retrieve_rrf(query, k_each=20, topk=20)
    r = rerank(query, f, topk=15)
    ctx = build_context(query, r, max_context_chars=3500)
    return ctx, r

def with_compression_only(query):
    f = retrieve_rrf(query, k_each=20, topk=20)
    # Stronger compression budget for this variant
    ctx = build_context(query, f, max_context_chars=1800)
    return ctx, f

def with_both(query):
    f = retrieve_rrf(query, k_each=20, topk=20)
    r = rerank(query, f, topk=15)
    ctx = build_context(query, r, max_context_chars=1800)
    return ctx, r

def ablate(queries: List[str]):
    rows = []
    for q in queries:
        t0 = time.time(); ctx_b, rb = baseline_only(q); t1 = time.time()
        t2 = time.time(); ctx_r, rr = with_rerank_only(q); t3 = time.time()
        t4 = time.time(); ctx_c, rc = with_compression_only(q); t5 = time.time()
        t6 = time.time(); ctx_bc, rbc = with_both(q); t7 = time.time()

        rows.append({
            "query": q,
            "baseline_ctx_chars": len(ctx_b),
            "+rerank_ctx_chars": len(ctx_r),
            "+compression_ctx_chars": len(ctx_c),
            "+both_ctx_chars": len(ctx_bc),
            "lat_baseline_s": round(t1 - t0, 3),
            "lat_+rerank_s": round(t3 - t2, 3),
            "lat_+compression_s": round(t5 - t4, 3),
            "lat_+both_s": round(t7 - t6, 3),
        })
    df = pd.DataFrame(rows)
    return df

sample_eval = [
    "How do I repair a small doorknob hole in drywall safely?",
    "What are steps to stop basement water seepage and seal cracks?",
    "How should I prepare and paint interior walls?",
]

ablation_results = ablate(sample_eval)
ablation_results


,query,baseline_ctx_chars,+rerank_ctx_chars,+compression_ctx_chars,+both_ctx_chars,lat_baseline_s,lat_+rerank_s,lat_+compression_s,lat_+both_s
0,How do I repair a small doorknob hole in drywa...,2700,2687,1739,1739,0.007,0.011,0.009,0.008
1,What are steps to stop basement water seepage ...,2741,2754,1798,1798,0.004,0.101,0.009,0.009
2,How should I prepare and paint interior walls?,2786,2604,1658,1658,0.003,0.009,0.008,0.008


## 11) Qualitative examples: wins & failures

In [12]:
def show_top(fused, n=3):
    for i, (s, ch) in enumerate(fused[:n], 1):
        print(f"{i:>2}. score={s:.4f}  {ch.doc_id}  chunk={ch.chunk_id}  -> {ch.source}")

q_wins = "How do I patch a small drywall hole and finish it flush?"
ans_w, ctx_w, logs_w, fused_w = run_pipeline(q_wins)
print("— WINS —"); show_top(fused_w, 5); print("\n", ans_w, "\nLogs:", logs_w)

q_fail = "How do I replace a specific model of smart thermostat that isn't in the manuals?"
ans_f, ctx_f, logs_f, fused_f = run_pipeline(q_fail)
print("\n— POTENTIAL FAILS —"); show_top(fused_f, 5); print("\n", ans_f, "\nLogs:", logs_f)
print("\nNote: If the answer requires brand-specific SKUs absent in the PDFs, expect lower recall/faithfulness.")


— WINS —
 1. score=0.0164  Complete home repair  with 350 projects and 2300 photos  chunk=16  -> /content/drive/MyDrive/Complete home repair  with 350 projects and 2300 photos.pdf
 2. score=0.0320  7 Different Ways to Repair Drywall  chunk=1  -> /content/drive/MyDrive/7 Different Ways to Repair Drywall.pdf
 3. score=0.0304  Complete home repair  with 350 projects and 2300 photos  chunk=150  -> /content/drive/MyDrive/Complete home repair  with 350 projects and 2300 photos.pdf
 4. score=0.0306  Complete home repair  with 350 projects and 2300 photos  chunk=27  -> /content/drive/MyDrive/Complete home repair  with 350 projects and 2300 photos.pdf
 5. score=0.0311  7 Different Ways to Repair Drywall  chunk=0  -> /content/drive/MyDrive/7 Different Ways to Repair Drywall.pdf

 Q: How do I patch a small drywall hole and finish it flush?

Relevant context (truncated):
[Complete home repair  with 350 projects and 2300 photos#$16] Let the spackle dry. Repairs/ Ceilings & Walls Patching Large Hole

## 12) Save results & config for your report

In [13]:
import json, pandas as pd, os, time
OUT_DIR = './week5_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

ablation_results.to_csv(os.path.join(OUT_DIR, 'ablation_results.csv'), index=False)

run_cfg = {
    "bm25": {"lib": "rank_bm25"},
    "dense": {"model": "sentence-transformers/all-MiniLM-L6-v2", "enabled": True},
    "fusion": "RRF(K=60)",
    "reranker": "cross-encoder/ms-marco-MiniLM-L-6-v2" if RERANK else "disabled",
    "mmr": {"lambda": 0.7},
    "compression": {"max_context_chars": 1800},
    "corpus_files": PDF_FILES,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}
with open(os.path.join(OUT_DIR, 'rag_adv_run_config.json'), 'w') as f:
    json.dump(run_cfg, f, indent=2)

print("Saved:", os.listdir(OUT_DIR))


Saved: ['rag_adv_run_config.json', 'ablation_results.csv']


## 13) (Optional) Guardrail hooks (citation enforcement / refusal)

In [14]:
def enforce_citations(answer: str) -> bool:
    # Require at least one [doc#chunk] tag in the draft
    return bool(re.search(r'\[[^\]]+#\$\d+\]', answer))

def safe_refusal_if_no_support(query: str, context: str) -> str:
    if not context.strip():
        return "Sorry — I couldn't find grounded information in the current corpus to answer that safely."
    return ""

# Example usage after generation:
draft = answer_with_context("What is the procedure to fix a crumpled corner bead?", "some [doc#chunk] context")
print("Has citations?", enforce_citations(draft))


Has citations? False


## Notes for your Week 5 report
- Compare **Baseline / +Rerank / +Compression / +Both** on a small eval set.
- Discuss **latency vs relevance** trade-offs. Cross-encoder reranking improves precision but adds latency.
- Highlight 2–3 qualitative wins and a failure case (e.g., brand-specific queries).  
- Ensure inline citations like `[Complete home repair  with 350 projects and 2300 photos#$42]` appear in the final answers.
